# Information Flow

Compare bidirectional, source-conditioned, and target-causal information flow.

This is the full **PyTorch** version of a challenge from [LLM Quest](https://bankoti.github.io/llm-quest). The in-browser challenge grades numpy; here you work with real tensors. Fill in each `TODO`, then run the checks cell.

Runs on the free Colab CPU runtime; switch to a GPU via *Runtime > Change runtime type* if you want to experiment at scale. PyTorch comes preinstalled.

In [ ]:
# Install the course package (vendored in the LLM Quest repo).
%pip install -q "mini-llm-course @ git+https://github.com/bankoti/llm-quest@main#subdirectory=colab/vendor"

In [ ]:
import torch

from mini_llm.architectures import BidirectionalEncoder, EncoderDecoderTransformer


torch.manual_seed(7)
encoder = BidirectionalEncoder(32, 8, 16, 4, 1, 32).eval()
tokens = torch.tensor([[1, 2, 3, 4]])
changed_future = tokens.clone()
changed_future[0, 3] = 9
with torch.no_grad():
    original, _ = encoder(tokens)
    changed, _ = encoder(changed_future)

In [ ]:
assert not torch.allclose(original[:, 0], changed[:, 0])

seq2seq = EncoderDecoderTransformer(32, 8, 16, 4, 1, 1, 32).eval()
source = torch.tensor([[1, 2, 3]])
target = torch.tensor([[4, 5, 6, 7]])
changed_target = target.clone()
changed_target[0, 3] = 8
changed_source = source.clone()
changed_source[0, 2] = 9
with torch.no_grad():
    base, _ = seq2seq(source, target)
    future_edit, _ = seq2seq(source, changed_target)
    source_edit, _ = seq2seq(changed_source, target)
assert torch.allclose(base[:, :3], future_edit[:, :3], atol=1e-6)
assert not torch.allclose(base[:, 0], source_edit[:, 0])
print("Architecture lab 01 passed")